# 1. Cleanup old processes

In [ ]:
!pkill -u "$USER" -f '^clingcon '
!ps -aux | grep clingcon

# 2. Run clingcon on instance with timeout

In [ ]:
import time, subprocess

timeMax = "86400"
instanceName = "green_circuit.lp"
coreCount = "64"

outfile = f"i_{instanceName}-t_{timeMax}-cpu_{coreCount}-{time.strftime('%Y%m%d_%H%M%S')}.log"
print(outfile)

sessionName = "asp"

inner_cmd = f"""
    bash -lc '
        echo "================ INSTANCE BEGIN ================" > logs/{outfile};
        echo "File: instances/{instanceName}" >> logs/{outfile};
        echo "------------------------------------------------" >> logs/{outfile};
        cat instances/{instanceName} >> logs/{outfile};
        echo >> logs/{outfile};
        echo "================ CLINGCON OUTPUT ================" >> logs/{outfile};
        systemd-run --user --scope -p MemoryMax=400G -p MemoryHigh=385G -p MemorySwapMax=0 \
        clingcon --stats model.lp game.lp instances/{instanceName} --opt-mode=ignore -n 1 --quiet=1 -t {coreCount} --time {timeMax} >> logs/{outfile} 2>&1
        '
"""

subprocess.run(["tmux", "new-session", "-d", "-s", sessionName, inner_cmd], check=True)
print(f"Attach with: tmux attach -d -t {sessionName}")

# 3. Visualize

In [ ]:
from clingcon_plotly_viewer import ClingconAssignmentViewer

viewer = ClingconAssignmentViewer.from_log("logs/i_red_circuit.lp-t_86400-cpu_64-20260322_193232.log")
#viewer = ClingconAssignmentViewer.from_log("logs/" + outfile)
ui = viewer.interactive_explorer_widget(cols=4, show_zeros_3d=True)
display(ui)